In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

In [22]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [3]:
import app.paths as paths
import src.load as load
import src.table as table
import src.graphs as graphs

In [4]:
c_preds, r_preds = load.load_preds()

In [6]:
meta,y = load.load_actuals()
transplant_weeks = meta['transplant_week'].values
idx_dict = load.create_idx_dict(meta)

In [7]:
season_c = table.shift_preds(c_preds,transplant_weeks)
season_r = table.shift_preds(r_preds,transplant_weeks)
season_act = table.shift_actuals(y,transplant_weeks)

In [8]:
diff = 2 * (season_c - season_r)
upper_preds = season_c + diff
lower_preds = season_r - diff
preds = np.stack([season_c, season_r,upper_preds,lower_preds],axis=-1)

In [9]:
upper = np.max(preds,axis=-1)
lower = np.min(preds,axis=-1)
std_dev = np.std(preds,axis=-1) / preds.shape[-1]**0.5
mean = np.mean(preds,axis=-1)
upper_mid = mean + std_dev
lower_mid = mean - std_dev

In [10]:
final_preds = {'upper':upper,'upper_mid':upper_mid,'mean':mean,'lower_mid':lower_mid,'lower':lower}

In [11]:
fig = graphs.this_season_CI_graph(final_preds,season_act)
fig.show()

In [23]:
df_preds = pd.DataFrame({
    'upper': final_preds['upper'].sum(axis=0)[0],
    'upper_mid': final_preds['upper_mid'].sum(axis=0)[0],
    'mean': final_preds['mean'].sum(axis=0)[0],
    'lower_mid': final_preds['lower_mid'].sum(axis=0)[0],
    'lower': final_preds['lower'].sum(axis=0)[0],
})


In [25]:
df_preds.T.astype(int)

,0,1,2,3,4,5,6,7,8,9,...,46,47,48,49,50,51,52,53,54,55
upper,0,0,0,0,0,0,0,0,0,176,...,61593,45133,38055,30200,21986,16307,12269,9525,9394,9406
upper_mid,0,0,0,0,0,0,0,0,0,82,...,39631,30220,24832,18938,14238,9461,6795,5380,5313,5318
mean,0,0,0,0,0,0,0,0,0,29,...,27248,21812,17377,12587,9870,5600,3708,3044,3012,3013
lower_mid,0,0,0,0,0,0,0,0,0,-23,...,14864,13403,9922,6237,5501,1740,622,707,711,708
lower,0,0,0,0,0,0,0,0,0,-117,...,-7097,-1509,-3299,-5025,-2246,-5105,-4851,-3436,-3369,-3379
